# Chapter 26 – Worked examples

This notebook runs the examples of [Chapter 26](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/0.2.00.2.0/part-4-optimization/ch26-experimental-methodology/) live. The code lives in the modules next to it
(`compare_runs.py`, `ga_experiment.py`, `rank_tests.py`, `performance_profile.py`, `toop_anytime.py`); the
cells only call them, so a change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q colorama==0.4.6 deepdiff==8.6.2 geojson==3.3.0 matplotlib networkx numpy packaging pandas pandera==0.34.1 scipy tqdm typing_extensions tzdata
%pip install -q --no-deps pandapower==3.5.6

In [ ]:
import matplotlib

matplotlib.use('Agg')

import compare_runs
import ga_experiment
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import performance_profile
import rank_tests
import toop_anytime

## Ten runs per configuration: rank-sum test and effect size

Independent samples, so the Mann–Whitney U test; $\hat A_{12}$ is the probability that a run of A scores lower
(better) than a run of B. On the synthetic data of the worked example it is 0.08: A loses almost always.

In [ ]:
rng = np.random.default_rng(0)
a = rng.normal(42.0, 3.0, size=10)
b = rng.normal(39.0, 3.0, size=10)
u, p, a12 = compare_runs.compare(a, b)
print(f'U={u:.1f}  p={p:.4f}  A12={a12:.2f}')
fig, ax = plt.subplots(figsize=(5, 3))
compare_runs.plot_ecdf({'A': a, 'B': b}, ax)
plt.close(fig)

## The lab's experiment, in small

Two GA configurations of Chapter 24 (crossover on or off), three seeds, 400 evaluations: one row per
(configuration, seed, evaluation), the median and quartile band against evaluations, the final values and the
full analysis (test, $\hat A_{12}$, bootstrap interval of the difference in medians).

In [ ]:
runs = ga_experiment.run_all(seeds=(0, 1, 2), budget=400)
print(runs.head(3).to_string())
finals = ga_experiment.final_values(runs)
print({k: np.round(v, 3).tolist() for k, v in finals.items()})
print(ga_experiment.analysis(finals['crossover'], finals['no crossover']))
fig, ax = plt.subplots(figsize=(6, 3.5))
ga_experiment.plot_anytime(runs, 'evaluation', ax)
plt.close(fig)

## Several configurations on several instances: Friedman, then Holm

Paired by instance, so Friedman over the configurations first; only if it rejects do the pairwise signed-rank
tests follow, with the Holm correction for the three comparisons.

In [ ]:
table = rank_tests.synthetic_instances()
statistic, p, ranks = rank_tests.friedman(table)
print(f'Friedman chi2 = {statistic:.2f}, p = {p:.4f}, mean ranks {ranks.round(2).to_dict()}')
print(rank_tests.holm(rank_tests.pairwise_signed_rank(table)).round(4).to_string())

## A performance profile

$\rho_s(\tau)$: the fraction of instances a solver solved within $\tau$ times the best time. $\rho_s(1)$ is the
fraction of instances on which it was fastest; for large $\tau$ the curve ends at the fraction it solved at all.

In [ ]:
times = performance_profile.synthetic_times()
print(performance_profile.profile(times, np.array([1.0, 2.0, 5.0, np.inf])).round(2).to_string())
fig, ax = plt.subplots(figsize=(5, 3))
performance_profile.plot_profile(times, ax=ax)
plt.close(fig)

## Five ToOp seeds, same `runtime_seconds`, different budgets

The committed table `toop_seeds.csv` comes from five runs of ToOp's small-grid example with
`random_seed` 1 to 5 and 30 s each. The number of epochs completed in those 30 s is the real budget, and it
varies by 44 % between runs on the same machine.

In [ ]:
seeds = pd.read_csv(toop_anytime.EXAMPLE)
print(toop_anytime.per_seed(seeds).to_string())
cut, common = toop_anytime.truncate(seeds)
print('smallest common epoch', common)